# Summit 2026 HOL — Build the Cortex Agent (API)
## Module 03: Create a Cortex Agent and Interact via REST API

_Icons used throughout this lab: 🛠️ Action  📌 Note  🔹 Info_

---

### What This Module Does:

1. 🛠️ Creates a **Cortex Agent** using the `CREATE AGENT` SQL command
2. 🛠️ Wires it to the semantic view (structured data) + Cortex Search (unstructured data)
3. 🛠️ Tests the agent with natural language questions
4. 🔹 Introduces **threads** for multi-turn conversations
5. 🛠️ Shows how to invoke the agent via **REST API** (for embedding in apps)

---

### What is a Cortex Agent?

🔹 A **Cortex Agent** is a Snowflake-native AI agent that orchestrates across structured and unstructured data to answer questions and take actions. It:

- **Plans** — Interprets intent, splits complex questions into subtasks
- **Uses tools** — Selects Cortex Analyst (SQL from semantic views) or Cortex Search (RAG from documents)
- **Reflects** — Evaluates results and iterates if needed
- **Responds** — Generates a grounded answer with citations

The Cortex Agent is the **developer-facing API** — you build it, configure it, and embed it in applications. In the next module, we'll see how **CoWork** provides a governed UI on top of the same agent for business users.

> **Documentation:** [Cortex Agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents)

---

### Estimated Time: **25–30 minutes**

### Prerequisites:
- Module 02 complete (Semantic View + Cortex Search Service created)

## Step 1: Set Context

In [ ]:
%%sql -r SetContext
USE ROLE NEXUS_ADMIN;
USE DATABASE NEXUS_HOL;
USE SCHEMA AGENTS;
USE WAREHOUSE NEXUS_WH;

## Step 2: Create the Cortex Agent

### Agent Anatomy

A Cortex Agent has these key components:

| Component | What It Does |
|---|---|
| **Model** | The LLM that powers reasoning (we use `auto` for best available) |
| **Tools** | Data sources the agent can call — Cortex Analyst for SQL, Cortex Search for RAG |
| **Tool Resources** | The specific semantic views and search services attached to each tool |
| **Instructions** | How the agent should behave, respond, and prioritize |
| **Sample Questions** | Conversation starters shown to users |

The `FROM SPECIFICATION` clause uses **YAML** to define the agent's configuration. This is the same spec whether you create via SQL, REST API, or the Snowsight UI.

> **Documentation:** [Build Agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/snowflake-intelligence/build-agents)

### 📌 Key Configuration: `tool_resources`

The `tool_resources` section is where you wire the agent's tools to actual Snowflake objects. Two critical connections:

1. **`semantic_view`** — The fully qualified name of the semantic view that Cortex Analyst will use to generate SQL. This is the "brain" 
of your structured data tool — without it, the agent has no understanding of your data model.

2. **`execution_environment`** — The compute that runs the SQL the agent generates. Cortex Agents are serverless — they don't inherit your
session's `USE WAREHOUSE` context. You must explicitly tell the agent which warehouse to use for query execution.

```yaml
tool_resources:
  nexus_analytics:
    semantic_view: "DB.SCHEMA.YOUR_SEMANTIC_VIEW"   # ← what data to query
    execution_environment:                          # ← where to run the SQL
      type: warehouse
      warehouse: "YOUR_WAREHOUSE"

📌 Why not just use the session warehouse? Because agents can be called from REST APIs, CoWork, and other contexts that don't have a SQL session. The execution environment must be self-contained in the agent spec.

In [ ]:
%%sql -r CreateAgent
CREATE OR REPLACE AGENT NEXUS_HOL.AGENTS.NEXUS_AGENT
  COMMENT = 'Nexus Capital AI Agent - answers questions about clients, portfolios, trades, and research'
  PROFILE = '{"display_name": "Nexus Capital Analyst", "color": "blue"}'
  FROM SPECIFICATION
$$
models:
  orchestration: auto

orchestration:
  budget:
    seconds: 45
    tokens: 16000

instructions:
  response: |
    You are the Nexus Capital AI Analyst. You help portfolio managers, relationship managers,
    and compliance officers understand client portfolios, trading activity, and market research.
    
    Guidelines:
    - Be concise and data-driven. Lead with numbers when available.
    - When showing financial data, format large numbers (e.g., $2.5B not $2500000000).
    - If a question spans both structured data (portfolios, trades) and unstructured data
      (research notes), use BOTH tools to provide a complete answer.
    - Always cite which data source your answer came from.
    - For compliance questions, flag any potential issues clearly.

  orchestration: |
    - For questions about client AUM, portfolio values, positions, or trades: use the Analyst tool.
    - For questions about market outlook, research opinions, or compliance reports: use the Search tool.
    - For questions that need both (e.g., "What is Sarah Chen's portfolio risk and are there any research notes about it?"), use both tools.

  sample_questions:
    - question: "Who are our top 5 clients by AUM?"
    - question: "What is our total technology sector exposure?"
    - question: "Are there any compliance concerns I should know about?"
    - question: "Show me recent trades over $1M and the rationale behind them."

tools:
  - tool_spec:
      type: "cortex_analyst_text_to_sql"
      name: "nexus_analytics"
      description: "Query structured financial data including client accounts, portfolio positions, trade history, and business metrics. Use this for any question about numbers, rankings, aggregations, or trends."
  - tool_spec:
      type: "cortex_search"
      name: "nexus_research"
      description: "Search through analyst research notes, market commentary, risk assessments, and compliance reports. Use this for qualitative insights, opinions, recommendations, and compliance flags."
  - tool_spec:
      type: "data_to_chart"
      name: "data_to_chart"
      description: "Generate visualizations from query results when the user asks for charts or visual breakdowns."

tool_resources:
  nexus_analytics:
    # ╔═══════════════════════════════════════════════════════════════════════════════════════════════════════════╗
    # ║  XXX: Replace with the fully qualified name of the semantic view you created in Module 02.                ║
    # ║  Hint: Format is DATABASE.SCHEMA.VIEW_NAME. Check the SHOW SEMANTIC VIEWS output from Module 02 Step 3.   ║
    # ╚═══════════════════════════════════════════════════════════════════════════════════════════════════════════╝
    semantic_view: "XXX"
    execution_environment:
      # ╔══════════════════════════════════════════════════════════════════════════════════════╗
      # ║  XXX: Replace with the warehouse name created in Module 01.                         ║
      # ║  Hint: This is the XS warehouse the agent will use to execute generated SQL.        ║
      # ║  Check your Setup notebook Step 3 or run: SHOW WAREHOUSES LIKE 'NEXUS%';            ║
      # ╚══════════════════════════════════════════════════════════════════════════════════════╝
      type: warehouse
      warehouse: "XXX"
  nexus_research:
    name: "NEXUS_HOL.AGENTS.NEXUS_RESEARCH_SEARCH"
    max_results: "5"
$$;

In [ ]:
%%sql -r ShowAgents
-- Verify agent was created
SHOW AGENTS IN SCHEMA NEXUS_HOL.AGENTS;

In [ ]:
%%sql -r DescribeAgent
-- Inspect the agent's full configuration
DESCRIBE AGENT NEXUS_HOL.AGENTS.NEXUS_AGENT;

## Step 3: Grant Access to the Agent

### Access Control for Cortex Agents

Cortex Agents use Snowflake's standard RBAC model. To call an agent, a role needs:
- `USAGE` on the agent object
- `USAGE` on the database and schema containing the agent
- The `SNOWFLAKE.CORTEX_USER` or `SNOWFLAKE.CORTEX_AGENT_USER` database role

Since we're using NEXUS_ADMIN (which already has CORTEX_USER), we just need to confirm access.

> **Documentation:** [Cortex Agents Access Control](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents#access-control-requirements)

In [ ]:
%%sql -r GrantAccess
-- Confirm our role has access (NEXUS_ADMIN owns the agent, so it has OWNERSHIP)
SHOW GRANTS ON AGENT NEXUS_HOL.AGENTS.NEXUS_AGENT;

### SQL vs. UI: Two Ways to Test

You can test the agent via **SQL** (using `DATA_AGENT_RUN` in the cells below) or via the **Agent Admin UI**:

**To test in the UI:**
1. Go to **AI & ML → Cortex AI → Agents → NEXUS_AGENT**
2. Click the **Test** tab (right side of the agent page)
3. Type your question in the chat box and hit enter

The UI is faster for iterative testing — no JSON formatting, instant responses in a chat interface. The SQL cells below prove the same 
thing works programmatically.

> **Recommendation:** Run Test 1 below via SQL to confirm `DATA_AGENT_RUN` works. Then switch to the Agent Admin **Test tab** for the remaining questions in Steps 5 and 6 — it's a better experience for rapid testing.

> **Documentation:** [DATA_AGENT_RUN](https://docs.snowflake.com/en/sql-reference/functions/data_agent_run-snowflake-cortex) | [Configure and Interact with Agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-manage)

## Step 4: Test the Agent — Structured Data (Cortex Analyst)

### Ask questions that require SQL generation from the semantic view

When you ask a quantitative question, the agent will:
1. Route to the `nexus_analytics` tool (Cortex Analyst)
2. Generate SQL from the semantic view
3. Execute the SQL
4. Summarize results in natural language

Let's test with a few structured data questions.

In [ ]:
%%sql -r dataframe_2
-- Test 1: Client ranking (uses Cortex Analyst → semantic view → SQL)
SELECT
  TRY_PARSE_JSON(
    SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
      'NEXUS_HOL.AGENTS.NEXUS_AGENT',
      $${
        "messages": [
          {
            "role": "user",
            "content": [
              { "type": "text", "text": "Who are our top 5 clients by AUM and what regions are they in?" }
            ]
          }
        ],
        "stream": false
      }$$
    )
  ) AS resp;

### A Note on `DATA_AGENT_RUN` Syntax

The function `SNOWFLAKE.CORTEX.DATA_AGENT_RUN` returns the agent's response as a string. You'll see three patterns in the wild:

| Pattern | Behavior | Best For |
|---------|----------|----------|
| `SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(...)` | Returns raw string | Quick testing — errors are visible as text |
| `SELECT PARSE_JSON(SNOWFLAKE.CORTEX.DATA_AGENT_RUN(...))` | Parses to JSON object; errors if response isn't valid JSON | Debugging — forces failures to surface |
| `SELECT TRY_PARSE_JSON(SNOWFLAKE.CORTEX.DATA_AGENT_RUN(...))` | Parses to JSON; returns NULL on bad JSON (never errors) | Structured output — makes it easy to extract specific fields from the response |

In Test 1 above, we use `TRY_PARSE_JSON` so the response renders as a structured JSON object in Snowsight (expandable, searchable). In the
remaining tests, we use the raw call for simplicity. Both work — choose based on whether you want structured output or quick readability.

In [ ]:
%%sql -r TestAgent2
-- Test 2: Portfolio analytics (multi-table join via semantic view)
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
    'NEXUS_HOL.AGENTS.NEXUS_AGENT',
    '{"messages": [{"role": "user", "content": [{"type": "text", "text": "Who are our top 5 clients by AUM?"}]}], "stream": false}'
    ) AS resp;

In [ ]:
%%sql -r TestAgent3
-- Test 3: Trade analysis
    SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
    'NEXUS_HOL.AGENTS.NEXUS_AGENT',
    '{"messages": [{"role": "user", "content": [{"type": "text", "text": "Show me all sell trades in the last 24 hours. What was the total volume sold?"}]}], "stream": false}'
    ) AS resp;

## Step 5: Test the Agent — Unstructured Data (Cortex Search)

### Ask questions that require searching research notes

When you ask a qualitative question, the agent will:
1. Route to the `nexus_research` tool (Cortex Search)
2. Perform semantic search across research notes
3. Retrieve relevant documents with citations
4. Synthesize an answer grounded in the research

In [ ]:
%%sql -r dataframe_4
-- Test 4: Research/RAG question (uses Cortex Search)
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'NEXUS_HOL.AGENTS.NEXUS_AGENT',
  '{"messages": [{"role": "user", "content": [{"type": "text", "text": "Are there any compliance concerns or risk warnings I should be aware of?"}]}], "stream": false}'
) AS resp;

In [ ]:
%%sql -r TestAgentSearch2
-- Test 5: Market outlook question (uses Cortex Search)
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'NEXUS_HOL.AGENTS.NEXUS_AGENT',
  '{"messages": [{"role": "user", "content": [{"type": "text", "text": "What is our research team saying about the semiconductor sector outlook?"}]}], "stream": false}'
) AS resp;

## Step 6: Test the Agent — Multi-Tool Orchestration

### Ask questions that require BOTH structured and unstructured data

This is where the agent's orchestration shines. It will:
1. Recognize the question spans multiple data types
2. Plan subtasks (structured query + document search)
3. Execute both tools
4. Synthesize a unified answer

This is something you can't do with a simple SQL query or a standalone RAG pipeline — it requires **agentic orchestration**.

In [ ]:
%%sql -r TestMultiTool
-- Test 6: Multi-tool question (requires both Analyst AND Search)
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'NEXUS_HOL.AGENTS.NEXUS_AGENT',
  '{"messages": [{"role": "user", "content": [{"type": "text", "text": "Show me Velocity Capital Partners current positions and any recent research notes about their holdings."}]}], "stream": false}'
  ) AS resp;

In [ ]:
%%sql -r TestMultiTool2
-- Test 7: Complex orchestration
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'NEXUS_HOL.AGENTS.NEXUS_AGENT',
  '{"messages": [{"role": "user", "content": [{"type": "text", "text": "Which clients have the highest unrealized gains in technology stocks, and what does our research say about the tech outlook? Should we be taking profits?"}]}], "stream": false}'
  ) AS resp;

---

## Step 7 (Optional): Evaluate and Improve the Agent

You have now tested the Nexus Agent with structured, unstructured, and multi-tool questions. Those manual tests help validate functionality, but they do not provide a repeatable way to measure and improve agent quality over time.

In this optional section, you'll explore Snowflake's **Self-Improving Agent** workflow, which combines:
- **Agent Evaluations** to measure agent quality
- **AI Observability**) to inspect traces and understand failures
- **CoCo** to recommend targeted improvements
- **Re-evaluation** to quantify whether those improvements worked

This pattern—introduced by Josh Reini and Elliott Botwick in [**Self-Improving Agents with CoCo**](https://www.snowflake.com/en/developers/guides/self-improving-agents-with-coco/)—creates a closed-loop process for continuously improving AI applications:

**Evaluate → Analyze → Improve → Re-evaluate → Measure**

Unlike traditional agent testing, this approach provides objective metrics, detailed execution traces, and a systematic path for improving agent behavior over time.

**What you'll do**
1. Create and register an evaluation dataset
2. Run Agent Evaluations against the Nexus Agent
3. Review evaluation scores and failure explanations
4. Inspect traces using AI Observability
5. Use CoCo to recommend agent improvements
6. Apply the improvements
7. Re-run the evaluation and compare results

**Other helpful resources** 
- Snowflake-internal doc for deeper [AI Observability enablement resources](https://docs.google.com/document/d/1DOt78YPC4h-8FI0WNFZLCeaopyIxAOH2Nm3yAMGeSz0/edit?usp=sharing)
- [Cortex Agent evaluations](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-evaluations)
- [AI Observability in Snowflake Cortex](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-observability)
- [EXECUTE_AI_EVALUATION](https://docs.snowflake.com/en/sql-reference/functions/execute_ai_evaluation)
- [GET_AI_EVALUATION_DATA](https://docs.snowflake.com/en/sql-reference/functions/get_ai_evaluation_data-snowflake-local)

🧭 *If you're short on time, skip to Step 8. However, this section demonstrates one of the most compelling capabilities in the Snowflake AI Data Cloud: a measurable, repeatable workflow for improving agent quality over time.*

In [ ]:
%%sql -r GrantEvalPrereqs
-- Run as ACCOUNTADMIN if these grants have not already been configured.
USE ROLE ACCOUNTADMIN;

GRANT CREATE DATASET ON SCHEMA NEXUS_HOL.AGENTS TO ROLE NEXUS_ADMIN;
GRANT EXECUTE TASK ON ACCOUNT TO ROLE NEXUS_ADMIN;

USE ROLE NEXUS_ADMIN;
USE DATABASE NEXUS_HOL;
USE SCHEMA AGENTS;
USE WAREHOUSE NEXUS_WH;

### 7.1 Create a small evaluation dataset

A good evaluation dataset should include the same patterns you just tested manually:
- structured questions answered through Cortex Analyst
- unstructured questions answered through Cortex Search
- multi-tool questions that require orchestration
- at least one guardrail or out-of-scope question

For a workshop, keep this intentionally small so attendees can see the full loop quickly.

📌  **Note:** The `GROUND_TRUTH` column must be a non-NULL `VARIANT` containing the key `ground_truth_output` for every row. If any row has a `NULL` or missing value, `answer_correctness` will skip that record and log "Missing ground truth." For evaluation approaches that don't require ground truth (such as `logical_consistency`), see Reza Brianca's article entitled [How to Evaluate Your Text-to-SQL Agent in Cortex Analyst Using TruLens](https://dev.to/reza_brianca/how-to-evaluate-your-text-to-sql-agent-in-cortex-analyst-using-trulens-28nc).

In [ ]:
CREATE OR REPLACE TABLE NEXUS_HOL.AGENTS.NEXUS_EVAL_DATASET (
  INPUT_QUERY VARCHAR,
  GROUND_TRUTH VARIANT,  -- Required: PARSE_JSON('{"ground_truth_output":"..."}') for answer_correctness
  TRACK VARCHAR
);

INSERT INTO NEXUS_HOL.AGENTS.NEXUS_EVAL_DATASET
  (INPUT_QUERY, GROUND_TRUTH, TRACK)
SELECT
  'Who are our top 5 clients by AUM and what regions are they in?',
  PARSE_JSON('{"ground_truth_output":"Return the top five clients by AUM, including each client region."}'),
  'structured'
UNION ALL SELECT
  'Which sectors have the largest exposure across our portfolios?',
  PARSE_JSON('{"ground_truth_output":"Summarize sector exposure using portfolio or position data, ranked by exposure."}'),
  'structured'
UNION ALL SELECT
  'What compliance concerns appear in the research notes?',
  PARSE_JSON('{"ground_truth_output":"Use the research notes to identify relevant compliance concerns and cite the basis from unstructured content."}'),
  'unstructured'
UNION ALL SELECT
  'Compare our technology exposure with the latest research commentary.',
  PARSE_JSON('{"ground_truth_output":"Use both structured portfolio exposure and unstructured research commentary in the answer."}'),
  'multi_tool'
UNION ALL SELECT
  'Can you book my vacation and tell me the weather in Paris?',
  PARSE_JSON('{"ground_truth_output":"Politely explain that the agent is designed for Nexus Capital portfolio and research questions and should not attempt unrelated travel actions."}'),
  'guardrail';

### 7.2 Register the evaluation dataset

Cortex Agent Evaluations runs against a registered Snowflake Dataset object rather than directly against the source table.

In this step, we'll create a Dataset object from the evaluation table and register it for use by the evaluation framework.

To make the notebook easy to rerun, the next SQL cell first removes any previously registered dataset with the same name and then recreates it from the latest contents of `NEXUS_EVAL_DATASET`.

After the dataset has been registered, we'll upload a YAML configuration file that references the registered dataset and defines the evaluation metrics to run.

In [ ]:
%%sql -r dataframe_11
DROP DATASET IF EXISTS NEXUS_HOL.AGENTS.NEXUS_EVAL_DS_V1;

CALL SYSTEM$CREATE_EVALUATION_DATASET(
  'Cortex Agent',
  'NEXUS_HOL.AGENTS.NEXUS_EVAL_DATASET',
  'NEXUS_HOL.AGENTS.NEXUS_EVAL_DS_V1',
  {
    'query_text': 'INPUT_QUERY',
    'ground_truth': 'GROUND_TRUTH'
  }
);

### 7.3 Upload the evaluation config and start the baseline run

The evaluation run reads a YAML configuration file from a stage.

Because the dataset was already registered in the previous step, this YAML **does not** include a `dataset:` block. It simply references the registered dataset and defines the target agent, run metadata, and metrics.

This baseline run is named `nexus_eval_run_1`. We'll compare against it after CoCo recommends improvements.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

session.sql("""
CREATE OR REPLACE FILE FORMAT NEXUS_HOL.AGENTS.EVAL_YAML_FORMAT
  TYPE = 'CSV'
  FIELD_DELIMITER = NONE
  RECORD_DELIMITER = '\n'
  SKIP_HEADER = 0
  FIELD_OPTIONALLY_ENCLOSED_BY = NONE
  ESCAPE_UNENCLOSED_FIELD = NONE
""").collect()

session.sql("""
CREATE STAGE IF NOT EXISTS NEXUS_HOL.AGENTS.EVAL_CONFIG_STAGE
  FILE_FORMAT = NEXUS_HOL.AGENTS.EVAL_YAML_FORMAT
""").collect()

yaml_text = """
evaluation:
  agent_params:
    agent_name: "NEXUS_HOL.AGENTS.NEXUS_AGENT"
    agent_type: "CORTEX AGENT"
  run_params:
    label: "baseline"
    description: "Nexus Capital Agent quality baseline"
  source_metadata:
    type: "dataset"
    dataset_name: "NEXUS_EVAL_DS_V1"

metrics:
  - "answer_correctness"
  - "logical_consistency"
""".strip()

config_path = "/tmp/nexus_agent_eval_config.yaml"
with open(config_path, "w") as f:
    f.write(yaml_text)

session.file.put(f"file://{config_path}", "@NEXUS_HOL.AGENTS.EVAL_CONFIG_STAGE", overwrite=True, auto_compress=False)
print("Uploaded @NEXUS_HOL.AGENTS.EVAL_CONFIG_STAGE/nexus_agent_eval_config.yaml")

In [ ]:
%%sql -r dataframe_5
CALL EXECUTE_AI_EVALUATION(
  'START',
  OBJECT_CONSTRUCT('run_name', 'nexus_eval_run_1'),
  '@NEXUS_HOL.AGENTS.EVAL_CONFIG_STAGE/nexus_agent_eval_config.yaml'
);

In [ ]:
%%sql -r CheckEvalStatus
-- Re-run this cell until the run reaches COMPLETED or FAILED.
CALL EXECUTE_AI_EVALUATION(
  'STATUS',
  OBJECT_CONSTRUCT('run_name', 'nexus_eval_run_1'),
  '@NEXUS_HOL.AGENTS.EVAL_CONFIG_STAGE/nexus_agent_eval_config.yaml'
);

### 7.4 Review scores and traces

Use the summary first, then inspect low-scoring records. In Snowsight, the evaluation record trace is the most customer-friendly way to explain *why* the agent selected tools and where it struggled.

Look for patterns such as:
- the wrong tool was selected
- only one tool was used when the question required both structured and unstructured data
- the answer was not grounded clearly enough in retrieved evidence
- the answer format was inconsistent or incomplete

In [ ]:
%%sql -r EvalSummary
SELECT
  METRIC_NAME,
  ROUND(AVG(EVAL_AGG_SCORE), 3) AS AVG_SCORE,
  COUNT(*) AS EVALUATED_RECORDS,
  ROUND(MIN(EVAL_AGG_SCORE), 3) AS LOWEST_SCORE
FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
  'NEXUS_HOL', 'AGENTS', 'NEXUS_AGENT', 'CORTEX AGENT', 'nexus_eval_run_1'
))
WHERE EVAL_AGG_SCORE IS NOT NULL
GROUP BY METRIC_NAME
ORDER BY AVG_SCORE ASC;

In [ ]:
%%sql -r dataframe_3
SELECT
  RECORD_ID,
  LEFT(INPUT, 120) AS QUESTION,
  METRIC_NAME,
  ROUND(EVAL_AGG_SCORE, 3) AS SCORE,
  e.VALUE:explanation::VARCHAR AS JUDGE_EXPLANATION
FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
  'NEXUS_HOL', 'AGENTS', 'NEXUS_AGENT', 'CORTEX AGENT', 'nexus_eval_run_1'
)),
LATERAL FLATTEN(INPUT => METRIC_CALLS) e
WHERE EVAL_AGG_SCORE IS NOT NULL
ORDER BY SCORE ASC, QUESTION;

In [ ]:
%%sql -r SnowsightEvalLink
SELECT
  'https://app.snowflake.com/'
  || LOWER(CURRENT_ORGANIZATION_NAME()) || '/'
  || LOWER(REPLACE(CURRENT_ACCOUNT_NAME(), '-', '_'))
  || '/#/agents/database/NEXUS_HOL/schema/AGENTS/agent/NEXUS_AGENT/evaluations/nexus_eval_run_1/records'
  AS SNOWSIGHT_EVALUATION_URL;

### 7.5 Use CoCo to improve the agent

Now use CoCo to turn the evaluation findings into targeted agent improvements.

Open CoCo in Snowsight. Start a new conversation. Provide the lowest-scoring records, judge explanations, and Snowsight trace observations from the previous step.

Suggested prompt:

> Review the evaluation results for `NEXUS_AGENT`.
>
> Identify recurring failure patterns and recommend improvements to the agent specification.
>
> Focus on:
> - orchestration instructions for when to use Analyst, Search, or both
> - response instructions for grounding, citations, and formatting
> - multi-tool workflows
> - guardrail handling for out-of-scope requests
>
> Return the recommended updates and explain why each update should improve the evaluation results.

This is the heart of the self-improving workflow: evaluations tell us *where* the agent struggled, traces show us *why*, and CoCo helps generate concrete changes.

In [ ]:
%%sql -r dataframe_6
-- Use this as context for CoCo. The agent specification must be replaced as a whole when modified.
DESCRIBE AGENT NEXUS_HOL.AGENTS.NEXUS_AGENT;

### 7.6 Use CoCo to improve the agent

Copy a few low-scoring evaluation records and their explanations from the previous step.

Paste them into CoCo along with the following prompt:

> Review the evaluation results for NEXUS_AGENT.
>
> Identify recurring failure patterns and recommend improvements to:
>
> - orchestration instructions
> - response instructions
>
> Focus on:
>
> - tool selection
> - multi-tool workflows
> - grounding responses in retrieved evidence
> - response quality and consistency
>
> Return the recommended instruction updates and explain why they should improve the evaluation scores.

Compare the recommendations against the evaluation explanations and determine which changes you would apply.

Important: updating the live agent specification replaces the full specification, so preserve the existing `models`, `orchestration`, `tools`, and `tool_resources` blocks and change only the instruction text that CoCo recommends.

For this HOL, the goal is not to hand-author YAML. The goal is to show that Snowflake can close the loop:

**Evaluation results -> trace inspection -> CoCo recommendations -> updated agent -> measured improvement**

After applying the CoCo recommendations, run the next cell to confirm the agent was updated before re-evaluating.

In [ ]:
%%sql -r dataframe_7
DESCRIBE AGENT NEXUS_HOL.AGENTS.NEXUS_AGENT;

### 7.7 Re-evaluate the agent

Run the evaluation again using the same dataset.

Reusing the same benchmark allows us to measure whether the CoCo recommendations improved agent quality.

In [ ]:
%%sql -r CompareEvalRuns
CALL EXECUTE_AI_EVALUATION(
  'START',
  OBJECT_CONSTRUCT('run_name', 'nexus_eval_run_2'),
  '@NEXUS_HOL.AGENTS.EVAL_CONFIG_STAGE/nexus_agent_eval_config.yaml'
);

### 7.8 Measure improvement

Compare the baseline run with the post-CoCo run.

**This is the payoff:** instead of relying on subjective before-and-after testing, we can quantify whether the agent improved.

**Evaluate -> Analyze -> Improve -> Re-evaluate -> Measure**

_____

💡 ***Notice that Agent Evaluations, AI Observability, CoCo, and the Agent itself all operate on the same platform.***

This allows teams to move seamlessly from identifying failures to generating improvements and measuring results without leaving Snowflake.

In [ ]:
%%sql -r dataframe_9
WITH baseline AS (
  SELECT METRIC_NAME, AVG(EVAL_AGG_SCORE) AS BASELINE_SCORE
  FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
    'NEXUS_HOL', 'AGENTS', 'NEXUS_AGENT', 'CORTEX AGENT', 'nexus_eval_run_1'
  ))
  WHERE EVAL_AGG_SCORE IS NOT NULL
  GROUP BY METRIC_NAME
), improved AS (
  SELECT METRIC_NAME, AVG(EVAL_AGG_SCORE) AS IMPROVED_SCORE
  FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
    'NEXUS_HOL', 'AGENTS', 'NEXUS_AGENT', 'CORTEX AGENT', 'nexus_eval_run_2'
  ))
  WHERE EVAL_AGG_SCORE IS NOT NULL
  GROUP BY METRIC_NAME
)
SELECT
  b.METRIC_NAME,
  ROUND(b.BASELINE_SCORE, 3) AS BASELINE_SCORE,
  ROUND(i.IMPROVED_SCORE, 3) AS IMPROVED_SCORE,
  ROUND(i.IMPROVED_SCORE - b.BASELINE_SCORE, 3) AS DELTA,
  CASE
    WHEN i.IMPROVED_SCORE > b.BASELINE_SCORE THEN 'improved'
    WHEN i.IMPROVED_SCORE < b.BASELINE_SCORE THEN 'regression'
    ELSE 'no change'
  END AS RESULT
FROM baseline b
JOIN improved i USING (METRIC_NAME)
ORDER BY DELTA DESC;

------------------------------------------------------------------
## Step 8: Understanding the REST API (For Application Embedding)

### When to Use the REST API

The SQL function `SNOWFLAKE.CORTEX.DATA_AGENT_RUN()` is great for testing in notebooks. But for production applications, you'll use the **REST API**:

- Embed the agent in a **Streamlit app**
- Call it from a **Python backend**
- Integrate with **external applications** via HTTP
- Maintain **threads** (multi-turn conversation context)

The REST API endpoint is:
```
POST /api/v2/databases/{database}/schemas/{schema}/agents/{name}:run
```

Here's an example request body:

```json
{
  "messages": [
    {
      "role": "user",
      "content": [
        {"type": "text", "text": "Who are our top clients by AUM?"}
      ]
    }
  ],
  "stream": false
}
```

And with a **thread** (for multi-turn conversations):

```json
{
  "thread_id": "your-thread-id",
  "messages": [
    {
      "role": "user",
      "content": [
        {"type": "text", "text": "Now show me just the ones in EMEA."}
      ]
}
```

Threads allow the agent to remember previous context without you re-sending the full history.

> **Documentation:** [Cortex Agents REST API](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents#interact-with-agents)

---

### Key Takeaway: Cortex Agent = Developer Building Block

The Cortex Agent is the **programmable foundation**. You:
- Define what data it can access (tools)
- Control how it behaves (instructions)
- Embed it anywhere (REST API)
- Maintain conversation state (threads)

In the next module, you'll see how **CoWork** puts a governed, no-code UI on top of this same agent — so business users can interact without writing code or API calls.

## Step 9: Try It — Python REST API Client

### Run this code LOCALLY (not in this notebook)

The REST API is how you embed agents in external applications — Streamlit apps, Python backends, CI/CD pipelines. This code runs
**outside** Snowflake (your laptop terminal, VS Code, or a local Jupyter notebook).

**Why not in this notebook?** Snowflake Notebooks run inside the platform's compute environment, which cannot call Snowflake's own REST
APIs via HTTP. The REST API is designed for external clients.

---

### Setup (one-time):

**1. Get your account URL** — run this in any Snowsight SQL worksheet:

In [ ]:
%%sql -r dataframe_1
SELECT CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME()
       || '.snowflakecomputing.com' AS account_url;

**2. Create/Use Programmatic Access Token (PAT)**

1. In Snowsight → click your name (bottom left) → Settings → Authentication → Programmatic Access Tokens
2. Click + Create Token
3. Set role to NEXUS_ADMIN
4. Copy the token (shown only once)

> ⚠️ If **"Programmatic Access Tokens"** is not in your menu, your HOL account does not have PAT enabled. Use Fallback option instead.

**Fallback Option: Username / Password**

Use the username and password you used to log into Snowsight. No additional setup required. The script handles this automatically when no PAT is provided.

---

**3. Install dependencies (if not already installed):**

```bash
pip3 install requests snowflake-connector-python
```

---

### Run the script:

The file `call_nexus_agent.py` is in [E2E_AI_HOL/assets/call_nexus_agent.py](../assets/call_nexus_agent.py).

Before running, open the file and fill in your values at the top:

```python
ACCOUNT_URL = "<your-account-url>"
PAT_TOKEN = "<your_pat_token>"

SNOWFLAKE_USER = "<your-username>"
SNOWFLAKE_PASSWORD = "<your-password>"
SNOWFLAKE_ACCOUNT = "<your-account>"
```

Then run from your terminal:

```bash
python3 /path/to/E2E_AI_HOL/assets/call_nexus_agent.py
```

The script will call your NEXUS_AGENT and print the response. Expect ~30–60 seconds for the agent to reason, generate SQL, execute, and respond.

> If using PAT, replace your_pat with your Programmatic Access Token. If you leave `PAT_TOKEN` empty, the script falls back to username/password via `DATA_AGENT_RUN`.

---

### What This Proves:

The same agent you tested in this notebook (via `DATA_AGENT_RUN`) and in CoWork (via the chat UI) is now callable from **any
external application** via Python. One agent, three delivery channels.

Documentation: [Cortex Agents REST API](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-run) | [Programmatic Access Tokens](https://docs.snowflake.com/en/user-guide/programmatic-access-tokens)

## ✅ Module 03 Complete!

### You now have:
- Cortex Agent: `NEXUS_HOL.AGENTS.NEXUS_AGENT`
  - Model: `auto` (best available, currently Claude 4 Sonnet)
  - Tool 1: `nexus_analytics` — Cortex Analyst on semantic view (structured queries)
  - Tool 2: `nexus_research` — Cortex Search on research notes (RAG)
  - Tool 3: `data_to_chart` — Visualization generation
  - Instructions tuned for financial analysis
  - Budget: 45 seconds, 16K tokens per request

### What You Proved:
- The agent correctly routes structured questions to Cortex Analyst
- The agent correctly routes unstructured questions to Cortex Search
- The agent can orchestrate across BOTH tools for complex questions
- The agent generates SQL from the semantic view (not raw table guessing)

---

### Cortex Agent vs. What's Next (CoWork)

| | Cortex Agent (this module) | CoWork (next module) |
|---|---|---|
| **Audience** | Developers, data engineers | Business users, analysts |
| **Interface** | SQL function or REST API | Chat UI in Snowsight |
| **How you use it** | Embed in apps, automations, pipelines | Open in browser, share via link |
| **MCP direction** | Expose agent TO external clients (Snowflake-managed MCP server) | Bring external tools INTO the agent (MCP connectors) |
| **Governance** | RBAC on agent object | SI-only users, scoped access, no raw SQL visibility |

Same underlying engine. Different delivery model. Both powered by the same semantic view and tools.

---

### Next Steps:

Continue to **Module 04: CoWork** — Deploy the same agent as a governed chat experience for business users.

**Optional extension covered:** Agent Evaluations baseline, trace review, guided improvement, and re-evaluation.